In [1]:
import sys
sys.path.append('../../../TaskExecutionTimeMining/')

import os
import shutil
import pickle
import numpy as np
from pathlib import Path
import subprocess

from divide_and_conquer_ib import *

In [2]:
model_path = "../../../../models/advanced/pcr/concept-name_seconds-in-day_day-of-week"
event_log_path = "../../transformed_event_logs/PCR_start_end_train.pickle"
screen_prefix = "PCR_csd_"


target_column = 'duration_seconds'
continuous_columns = [
    'seconds_in_day',
    #'case:RequestedAmount_start'
]
categorical_columns = [
    'concept:name',
    #'org:resource_start',
    'day_of_week'
]


In [3]:
with open(event_log_path, "rb") as f:
    event_log = pickle.load(f)

transformed_event_log = event_log.copy()


/tmp/ipykernel_2778606/2883312924.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  event_log = pickle.load(f)


In [4]:
with open(event_log_path, "rb") as f:
    event_log = pickle.load(f)

transformed_event_log = event_log.copy()

transformations = dict()
for num_attr in continuous_columns + [target_column]:
    transformed_event_log[num_attr] = np.log1p(transformed_event_log[num_attr]+1)
    m = transformed_event_log[num_attr].mean()
    std = transformed_event_log[num_attr].std()
    transformed_event_log[num_attr] = (transformed_event_log[num_attr] - m) / std
    transformations[num_attr] = (m, std)

/tmp/ipykernel_2778606/25019715.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  event_log = pickle.load(f)


In [5]:
# clear the model directory
ignore_file = "drbart_variable.r"

for entry in os.listdir(model_path):
    if entry == ignore_file:
        continue  # Skip this file
    path = os.path.join(model_path, entry)
    if os.path.isfile(path) or os.path.islink(path):
        os.unlink(path)  # Remove file or symlink
    elif os.path.isdir(path):
        shutil.rmtree(path)  # Remove directory and all contents


In [6]:
res = divide_and_conquer_ib(
    transformed_event_log,
    target_column=target_column,
    continuous_columns=continuous_columns,
    categorical_columns=categorical_columns,
    n_clusters=32
)

Clustering concept:name
X discrete (categorical), Adjusted n_bins_x: 8, Max X_d index: 7, Unique X_d bins: 8
Y continuous (quantile bins), Adjusted n_bins_y: 512, Max Y_d index: 511, Unique Y_d bins: 512


/home/LordKunkler/.local/share/virtualenvs/TaskExecutionTimeMining-yRnjZRF7/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


Initial (Clusters: 8): Mutual Information I(T; Y) = 1.731252


Merging clusters: 0it [00:00, ?it/s]

New best MI: 1.7312523187864803 for column concept:name
Clustering day_of_week
X discrete (categorical), Adjusted n_bins_x: 7, Max X_d index: 6, Unique X_d bins: 7
Y continuous (quantile bins), Adjusted n_bins_y: 512, Max Y_d index: 511, Unique Y_d bins: 512
Initial (Clusters: 7): Mutual Information I(T; Y) = 0.481377
Clustering seconds_in_day
X continuous (quantile bins), Adjusted n_bins_x: 128, Max X_d index: 127, Unique X_d bins: 128
Y continuous (quantile bins), Adjusted n_bins_y: 512, Max Y_d index: 511, Unique Y_d bins: 512
Initial (Clusters: 128): Mutual Information I(T; Y) = 3.133766


...

Iteration 94 (Clusters: 34): Mutual Information I(T; Y) = 1.951430


Iteration 95 pairs:   0%|          | 0/561 [00:00<?, ?it/s]

Iteration 95 (Clusters: 33): Mutual Information I(T; Y) = 1.927419


Iteration 96 pairs:   0%|          | 0/528 [00:00<?, ?it/s]

Iteration 96 (Clusters: 32): Mutual Information I(T; Y) = 1.902614
New best MI: 1.9026140497821773 for column seconds_in_day


In [7]:
# General version
unique_ids = np.unique(res[1])

# Create a dictionary of DataFrames
divided_event_logs = {uid: transformed_event_log[res[1] == uid].reset_index(drop=True) for uid in unique_ids}

In [8]:
# Assume routed_dfs is your dictionary of DataFrames
base_dir = Path(model_path)  # or any base directory name you like
base_dir.mkdir(exist_ok=True)

with open(base_dir / "gate.pickle", "wb") as f:
    pickle.dump(res, f)

with open(base_dir / "transformations.pickle", "wb") as f:
    pickle.dump(transformations, f)

for uid, sub_event_log in divided_event_logs.items():
    folder = base_dir / str(uid)
    folder.mkdir(exist_ok=True)
    sub_event_log.to_csv(folder / "data.csv", index=False)
    shutil.copy(
        model_path + "/drbart_variable.r",
        folder / "drbart_variable.r"
    )
    cmd = f"screen -dmS {screen_prefix+str(uid)} bash -c 'cd \"{folder}\" && ../../../../../src/DRBartModelTrainer/train_dr_bart.sh ../../../../../src/TaskExecutionTimeMining/drbart_training/numerical_stable_model/drbart_static.r > out.log 2>&1'"
    print(cmd)
    subprocess.run(cmd, shell=True)
    print(uid, sub_event_log.shape)

screen -dmS PCR_csd_0 bash -c 'cd "../../../../models/advanced/pcr/concept-name_seconds-in-day_day-of-week/0" && ../../../../../src/DRBartModelTrainer/train_dr_bart.sh ../../../../../src/TaskExecutionTimeMining/drbart_training/numerical_stable_model/drbart_static.r > out.log 2>&1'
0 (1070, 42)
screen -dmS PCR_csd_1 bash -c 'cd "../../../../models/advanced/pcr/concept-name_seconds-in-day_day-of-week/1" && ../../../../../src/DRBartModelTrainer/train_dr_bart.sh ../../../../../src/TaskExecutionTimeMining/drbart_training/numerical_stable_model/drbart_static.r > out.log 2>&1'
1 (920, 42)
screen -dmS PCR_csd_2 bash -c 'cd "../../../../models/advanced/pcr/concept-name_seconds-in-day_day-of-week/2" && ../../../../../src/DRBartModelTrainer/train_dr_bart.sh ../../../../../src/TaskExecutionTimeMining/drbart_training/numerical_stable_model/drbart_static.r > out.log 2>&1'
2 (834, 42)
screen -dmS PCR_csd_3 bash -c 'cd "../../../../models/advanced/pcr/concept-name_seconds-in-day_day-of-week/3" && ../.